# ODI to Databricks Migration

## Source File: `SCEN_TASK_NO_inventory_product_dim_etl.sql`
**Converted On:** 2024-07-30

**Description:** This notebook processes inventory product dimension data, updating existing records and inserting new ones into `W_INVENTORY_PRODUCT_D`.
It includes category attribute updates, flow table population, and final merge operations, along with ETL load date management.

In [ ]:
dbutils.widgets.text("BIAPPS_DATASOURCE_NUM_ID", "999", "BIAPPS_DATASOURCE_NUM_ID")
dbutils.widgets.text("BIAPPS_WH_DATASOURCE_NUM_ID", "0", "BIAPPS_WH_DATASOURCE_NUM_ID")
dbutils.widgets.text("BIAPPS_ETL_USAGE_CODE", "__NOT_APPLICABLE__", "BIAPPS_ETL_USAGE_CODE")
dbutils.widgets.text("BIAPPS_LOW_DATE", "1900-01-01 00:00:00", "BIAPPS_LOW_DATE")
dbutils.widgets.text("BIAPPS_SOURCE_CODE", "GL", "BIAPPS_SOURCE_CODE")
dbutils.widgets.text("BIAPPS_TARGET_CODE", "GL", "BIAPPS_TARGET_CODE")
dbutils.widgets.text("BIAPPS_IS_INCREMENTAL", "Y", "BIAPPS_IS_INCREMENTAL")
dbutils.widgets.text("BIAPPS_ETL_PROC_WID", "12345", "BIAPPS_ETL_PROC_WID")
dbutils.widgets.text("BIAPPS_EXECUTION_ID", "67890", "BIAPPS_EXECUTION_ID")
dbutils.widgets.text("BIAPPS_PRUNE_DAYS", "0", "BIAPPS_PRUNE_DAYS")
dbutils.widgets.text("ODI_SESS_NO", "3260538", "ODI_SESS_NO")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_bi_apps_params AS
SELECT
  CAST('${BIAPPS_DATASOURCE_NUM_ID}' AS BIGINT) AS BIAPPS_DATASOURCE_NUM_ID,
  CAST('${BIAPPS_WH_DATASOURCE_NUM_ID}' AS BIGINT) AS BIAPPS_WH_DATASOURCE_NUM_ID,
  '${BIAPPS_ETL_USAGE_CODE}' AS BIAPPS_ETL_USAGE_CODE,
  to_timestamp(substring('${BIAPPS_LOW_DATE}', 1, 19), 'yyyy-MM-dd HH:mm:ss') AS BIAPPS_LOW_DATE,
  '${BIAPPS_SOURCE_CODE}' AS BIAPPS_SOURCE_CODE,
  '${BIAPPS_TARGET_CODE}' AS BIAPPS_TARGET_CODE,
  '${BIAPPS_IS_INCREMENTAL}' AS BIAPPS_IS_INCREMENTAL,
  CAST('${BIAPPS_ETL_PROC_WID}' AS BIGINT) AS BIAPPS_ETL_PROC_WID,
  CAST('${BIAPPS_EXECUTION_ID}' AS BIGINT) AS BIAPPS_EXECUTION_ID,
  CAST('${BIAPPS_PRUNE_DAYS}' AS INT) AS BIAPPS_PRUNE_DAYS,
  '${ODI_SESS_NO}' AS ODI_SESS_NO;

In [ ]:
display(spark.sql("SELECT * FROM v_bi_apps_params"))

## Check ETL Load Dates (SCEN_TASK_NO {1})

In [ ]:
%sql
SELECT
  (CASE
    WHEN COUNT(*) > 0
    THEN 'Y'
    ELSE 'N'
  END)
FROM
  workspace.prxbi_dw.w_etl_load_dates
WHERE
  package_name = 'SILOS_SIL_INVENTORYPRODUCTDIMENSION'
  AND (
    datasource_num_id = (SELECT BIAPPS_DATASOURCE_NUM_ID FROM v_bi_apps_params)
    OR datasource_num_id = (SELECT BIAPPS_WH_DATASOURCE_NUM_ID FROM v_bi_apps_params)
  )
  AND etl_usage_code = (SELECT BIAPPS_ETL_USAGE_CODE FROM v_bi_apps_params)
  AND committed = '1';

## Merge Product Category Attributes (SCEN_TASK_NO {2})

In [ ]:
%sql
MERGE INTO workspace.prxbi_dw.w_inventory_product_d AS T
USING (
  SELECT DISTINCT
    X.integration_id,
    X.inv_prod_cat1,
    Y.inv_prod_cat1_wid
  FROM
    (
      SELECT
        B.integration_id,
        A.integration_id AS inv_prod_cat1
      FROM
        workspace.prxbi_dw.w_ora_invitem_category_tmp AS A
      INNER JOIN
        workspace.prxbi_dw.w_inventory_product_d AS B
      ON
        CONCAT_WS('~', A.inventory_item_id, A.organization_id) = B.integration_id
      WHERE
        A.integration_id <> B.inv_prod_cat1
    ) AS X,
    (
      SELECT
        P.integration_id,
        Q.row_wid AS inv_prod_cat1_wid
      FROM
        workspace.prxbi_dw.w_ora_invitem_category_tmp AS P
      INNER JOIN
        workspace.prxbi_dw.w_prod_cat_dh AS Q
      ON
        Q.integration_id = P.integration_id
    ) AS Y
  WHERE
    X.inv_prod_cat1 = Y.integration_id
) AS S
ON
  T.integration_id = S.integration_id
WHEN MATCHED THEN
  UPDATE SET
    T.inv_prod_cat1 = S.inv_prod_cat1,
    T.inv_prod_cat1_wid = S.inv_prod_cat1_wid;

## ODI Incremental Update Setup (SCEN_TASK_NO {30})

In [ ]:
%sql
-- SCEN_TASK_NO {30}: ODI Incremental Update Metadata
-- IKM BIAPPS Oracle Incremental Update 11.1.1.11.1.20181011
-- KM Options and Behaviour:
-- Uses Change Indicator: Y, DATASOURCE_NUM_ID column is part of the Unique Key
-- ROW_WID column present: Y, Bulk Mode Option: N, DELETE_FLG column present: Y
-- Error Logging Option: N, Setting error table name to E$_3260538_1
-- Error Logging Supported: Y, Is this a Dim ETL: Y, Auto Correction Enabled: Y
-- Bypass Flow Table: N, Target Type: OTHER
-- Work Objects:
-- Setting flow table name to PRXBI_DW.I$_3260538_1
-- Flex allocation change detection logic: N

## Error Table Setup

In [ ]:
%sql
-- SCEN_TASK_NO {60}: Drop error table
DROP TABLE IF EXISTS workspace.prxbi_dw.e_inventory_product_d;

In [ ]:
%sql
-- SCEN_TASK_NO {70}: Create error table
CREATE TABLE workspace.prxbi_dw.e_inventory_product_d
(
  ora_err_number            DOUBLE,
  ora_err_mesg              STRING,
  ora_err_rowid             STRING,
  ora_err_optyp             STRING,
  ora_err_tag               STRING,
  ind_update                STRING,
  diagnostic_rowid          STRING,
  error_type_ind            STRING,
  autocorrect_ind           STRING DEFAULT 'N',
  autocorrect_code          STRING,
  autocorrect_desc          STRING,
  committed                 STRING DEFAULT '0',
  row_wid                   STRING,
  product_wid               STRING,
  inventory_org_wid         STRING,
  plant_loc_wid             STRING,
  product_num               STRING,
  abc_ind                   STRING,
  planner_code              STRING,
  procurement_type_code     STRING,
  spc_proc_type_code        STRING,
  buyer_code                STRING,
  buyer_name                STRING,
  commodity_code            STRING,
  commodity_uom_code        STRING,
  profit_center_num         STRING,
  reorder_point             STRING,
  safety_stock_level        STRING,
  min_lot_size              STRING,
  max_lot_size              STRING,
  fixed_lot_size            STRING,
  max_stock_level           STRING,
  lot_ordering_cost         STRING,
  mrp_time_fence            STRING,
  ext_procure_time          STRING,
  internal_mfg_time         STRING,
  max_storage_days          STRING,
  mrp_profile_code          STRING,
  mrp_type_code             STRING,
  mrp_grp_code              STRING,
  lot_size_code             STRING,
  backflush_ind             STRING,
  qa_inspect_ind            STRING,
  repetitive_mfg_ind        STRING,
  bulk_item_ind             STRING,
  forecast_period           STRING,
  mfg_uom_code              STRING,
  issue_uom_code            STRING,
  manufacturing_place       STRING,
  loading_type_code         STRING,
  int_store_loc_code        STRING,
  ext_store_loc_code        STRING,
  active_flg                STRING,
  created_by_wid            STRING,
  changed_by_wid            STRING,
  created_on_dt             STRING,
  changed_on_dt             STRING,
  aux1_changed_on_dt        STRING,
  aux2_changed_on_dt        STRING,
  aux3_changed_on_dt        STRING,
  aux4_changed_on_dt        STRING,
  src_eff_from_dt           STRING,
  src_eff_to_dt             STRING,
  effective_from_dt         STRING,
  current_flg               STRING,
  w_insert_dt               STRING,
  w_update_dt               STRING,
  datasource_num_id         STRING,
  etl_proc_wid              STRING,
  integration_id            STRING,
  tenant_id                 STRING,
  x_custom                  STRING,
  inv_prod_cat1             STRING,
  inv_prod_cat2             STRING,
  inv_prod_cat3             STRING,
  inv_prod_cat4             STRING,
  inv_prod_cat5             STRING,
  inv_prod_cat6             STRING,
  inv_prod_cat7             STRING,
  inv_prod_cat8             STRING,
  inv_prod_cat9             STRING,
  inv_prod_cat10            STRING,
  inv_prod_cat1_wid         STRING,
  inv_prod_cat2_wid         STRING,
  inv_prod_cat3_wid         STRING,
  inv_prod_cat4_wid         STRING,
  inv_prod_cat5_wid         STRING,
  inv_prod_cat6_wid         STRING,
  inv_prod_cat7_wid         STRING,
  inv_prod_cat8_wid         STRING,
  inv_prod_cat9_wid         STRING,
  inv_prod_cat10_wid        STRING,
  invoiceable_item_flag     STRING,
  invoice_enabled_flag      STRING,
  primary_uom_code          STRING,
  c_primary_uom_code        STRING,
  unspsc_code               STRING,
  unspsc_inv_prod_cat_wid   STRING,
  commodity_name            STRING,
  commodity_uom_name        STRING,
  ext_store_loc_name        STRING,
  int_store_loc_name        STRING,
  issue_uom_name            STRING,
  loading_type_name         STRING,
  lot_size_name             STRING,
  mfg_uom_name              STRING,
  mrp_grp_name              STRING,
  mrp_profile_name          STRING,
  mrp_type_name             STRING,
  mrp_type_name             STRING,
  planner_name              STRING,
  primary_uom_name          STRING,
  procurement_type_name     STRING,
  profit_center_name        STRING,
  spc_proc_type_name        STRING,
  status_code               STRING,
  w_status_code             STRING,
  product_type_code         STRING,
  make_buy_ind              STRING,
  fixed_lead_time           STRING,
  variable_lead_time        STRING,
  cumulative_total_lead_time STRING,
  postprocessing_lead_time  STRING,
  preprocessing_lead_time   STRING,
  process_quality_enabled_flg STRING,
  x_price_sequence          STRING,
  x_organization_name       STRING,
  x_product_desc            STRING,
  x_uom_desc                STRING,
  x_inv_item_flg            STRING,
  x_stock_item_flg          STRING,
  x_trans_flg               STRING,
  x_rev_flg                 STRING,
  x_cost_flg                STRING,
  x_gcoa_acct               STRING,
  x_gcoa_prod               STRING,
  x_tax_cat                 STRING,
  organization_id           STRING,
  x_gcoa_loc_acct           STRING,
  delete_flg                STRING
) USING DELTA;

## Flow Table Setup

In [ ]:
%sql
-- SCEN_TASK_NO {110}: Drop flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_inventory_product_d_flow;

In [ ]:
%sql
-- SCEN_TASK_NO {120}: Create flow table
CREATE TABLE workspace.prxbi_dw.i_inventory_product_d_flow
(
  src_eff_from_dt           TIMESTAMP,
  datasource_num_id         BIGINT,
  integration_id            STRING,
  row_wid                   BIGINT,
  product_wid               BIGINT,
  inventory_org_wid         BIGINT,
  plant_loc_wid             BIGINT,
  product_num               STRING,
  abc_ind                   STRING,
  planner_code              STRING,
  procurement_type_code     STRING,
  spc_proc_type_code        STRING,
  buyer_code                STRING,
  buyer_name                STRING,
  commodity_code            STRING,
  commodity_uom_code        STRING,
  profit_center_num         STRING,
  reorder_point             DOUBLE,
  safety_stock_level        DOUBLE,
  min_lot_size              DOUBLE,
  max_lot_size              DOUBLE,
  fixed_lot_size            DOUBLE,
  max_stock_level           DOUBLE,
  lot_ordering_cost         DOUBLE,
  mrp_time_fence            DOUBLE,
  ext_procure_time          DOUBLE,
  internal_mfg_time         DOUBLE,
  max_storage_days          DOUBLE,
  mrp_profile_code          STRING,
  mrp_type_code             STRING,
  mrp_grp_code              STRING,
  lot_size_code             STRING,
  backflush_ind             STRING,
  qa_inspect_ind            STRING,
  repetitive_mfg_ind        STRING,
  bulk_item_ind             STRING,
  forecast_period           STRING,
  mfg_uom_code              STRING,
  issue_uom_code            STRING,
  manufacturing_place       STRING,
  loading_type_code         STRING,
  int_store_loc_code        STRING,
  ext_store_loc_code        STRING,
  active_flg                STRING,
  created_by_wid            BIGINT,
  changed_by_wid            BIGINT,
  created_on_dt             TIMESTAMP,
  changed_on_dt             TIMESTAMP,
  aux1_changed_on_dt        TIMESTAMP,
  aux2_changed_on_dt        TIMESTAMP,
  aux3_changed_on_dt        TIMESTAMP,
  aux4_changed_on_dt        TIMESTAMP,
  src_eff_to_dt             TIMESTAMP,
  effective_from_dt         TIMESTAMP,
  effective_to_dt           TIMESTAMP,
  delete_flg                STRING,
  current_flg               STRING,
  w_insert_dt               TIMESTAMP,
  w_update_dt               TIMESTAMP,
  etl_proc_wid              BIGINT,
  tenant_id                 STRING,
  x_custom                  STRING,
  inv_prod_cat1             STRING,
  inv_prod_cat2             STRING,
  inv_prod_cat3             STRING,
  inv_prod_cat4             STRING,
  inv_prod_cat5             STRING,
  inv_prod_cat6             STRING,
  inv_prod_cat7             STRING,
  inv_prod_cat8             STRING,
  inv_prod_cat9             STRING,
  inv_prod_cat10            STRING,
  inv_prod_cat1_wid         BIGINT,
  inv_prod_cat2_wid         BIGINT,
  inv_prod_cat3_wid         BIGINT,
  inv_prod_cat4_wid         BIGINT,
  inv_prod_cat5_wid         BIGINT,
  inv_prod_cat6_wid         BIGINT,
  inv_prod_cat7_wid         BIGINT,
  inv_prod_cat8_wid         BIGINT,
  inv_prod_cat9_wid         BIGINT,
  inv_prod_cat10_wid        BIGINT,
  invoiceable_item_flag     STRING,
  invoice_enabled_flag      STRING,
  primary_uom_code          STRING,
  c_primary_uom_code        STRING,
  unspsc_code               STRING,
  unspsc_inv_prod_cat_wid   BIGINT,
  commodity_name            STRING,
  commodity_uom_name        STRING,
  ext_store_loc_name        STRING,
  int_store_loc_name        STRING,
  issue_uom_name            STRING,
  loading_type_name         STRING,
  lot_size_name             STRING,
  mfg_uom_name              STRING,
  mrp_grp_name              STRING,
  mrp_profile_name          STRING,
  mrp_type_name             STRING,
  planner_name              STRING,
  primary_uom_name          STRING,
  procurement_type_name     STRING,
  profit_center_name        STRING,
  spc_proc_type_name        STRING,
  status_code               STRING,
  w_status_code             STRING,
  product_type_code         STRING,
  make_buy_ind              STRING,
  fixed_lead_time           DOUBLE,
  variable_lead_time        DOUBLE,
  cumulative_total_lead_time DOUBLE,
  postprocessing_lead_time  DOUBLE,
  preprocessing_lead_time   DOUBLE,
  process_quality_enabled_flg STRING,
  x_price_sequence          STRING,
  x_organization_name       STRING,
  x_product_desc            STRING,
  x_uom_desc                STRING,
  x_inv_item_flg            STRING,
  x_stock_item_flg          STRING,
  x_trans_flg               STRING,
  x_rev_flg                 STRING,
  x_cost_flg                STRING,
  x_gcoa_acct               STRING,
  x_gcoa_prod               STRING,
  x_tax_cat                 STRING,
  organization_id           STRING,
  x_gcoa_loc_acct           STRING,
  ind_update                STRING
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {130}: Insert into flow table
INSERT INTO workspace.prxbi_dw.i_inventory_product_d_flow
(
  product_wid,
  inventory_org_wid,
  plant_loc_wid,
  product_num,
  abc_ind,
  planner_code,
  procurement_type_code,
  spc_proc_type_code,
  buyer_code,
  buyer_name,
  commodity_code,
  commodity_uom_code,
  profit_center_num,
  reorder_point,
  safety_stock_level,
  min_lot_size,
  max_lot_size,
  fixed_lot_size,
  max_stock_level,
  lot_ordering_cost,
  mrp_time_fence,
  ext_procure_time,
  internal_mfg_time,
  max_storage_days,
  mrp_profile_code,
  mrp_type_code,
  mrp_grp_code,
  lot_size_code,
  backflush_ind,
  qa_inspect_ind,
  repetitive_mfg_ind,
  bulk_item_ind,
  forecast_period,
  mfg_uom_code,
  issue_uom_code,
  manufacturing_place,
  loading_type_code,
  int_store_loc_code,
  ext_store_loc_code,
  active_flg,
  created_by_wid,
  changed_by_wid,
  created_on_dt,
  changed_on_dt,
  aux1_changed_on_dt,
  aux2_changed_on_dt,
  aux3_changed_on_dt,
  aux4_changed_on_dt,
  src_eff_from_dt,
  src_eff_to_dt,
  effective_from_dt,
  delete_flg,
  datasource_num_id,
  integration_id,
  tenant_id,
  x_custom,
  inv_prod_cat1,
  inv_prod_cat2,
  inv_prod_cat3,
  inv_prod_cat4,
  inv_prod_cat5,
  inv_prod_cat6,
  inv_prod_cat7,
  inv_prod_cat8,
  inv_prod_cat9,
  inv_prod_cat10,
  inv_prod_cat1_wid,
  inv_prod_cat2_wid,
  inv_prod_cat3_wid,
  inv_prod_cat4_wid,
  inv_prod_cat5_wid,
  inv_prod_cat6_wid,
  inv_prod_cat7_wid,
  inv_prod_cat8_wid,
  inv_prod_cat9_wid,
  inv_prod_cat10_wid,
  invoiceable_item_flag,
  invoice_enabled_flag,
  primary_uom_code,
  c_primary_uom_code,
  unspsc_code,
  unspsc_inv_prod_cat_wid,
  commodity_name,
  commodity_uom_name,
  ext_store_loc_name,
  int_store_loc_name,
  issue_uom_name,
  loading_type_name,
  lot_size_name,
  mfg_uom_name,
  mrp_grp_name,
  mrp_profile_name,
  mrp_type_name,
  planner_name,
  primary_uom_name,
  procurement_type_name,
  profit_center_name,
  spc_proc_type_name,
  status_code,
  w_status_code,
  product_type_code,
  make_buy_ind,
  fixed_lead_time,
  variable_lead_time,
  cumulative_total_lead_time,
  postprocessing_lead_time,
  preprocessing_lead_time,
  process_quality_enabled_flg,
  x_price_sequence,
  x_organization_name,
  x_product_desc,
  x_uom_desc,
  x_inv_item_flg,
  x_stock_item_flg,
  x_trans_flg,
  x_rev_flg,
  x_cost_flg,
  x_gcoa_acct,
  x_gcoa_prod,
  x_tax_cat,
  organization_id,
  x_gcoa_loc_acct,
  current_flg,
  effective_to_dt,
  ind_update
)
SELECT
  C.product_wid,
  C.inventory_org_wid,
  C.plant_loc_wid,
  C.product_num,
  C.abc_ind,
  C.planner_code,
  C.procurement_type_code,
  C.spc_proc_type_code,
  C.buyer_code,
  C.buyer_name,
  C.commodity_code,
  C.commodity_uom_code,
  C.profit_center_num,
  C.reorder_point,
  C.safety_stock_level,
  C.min_lot_size,
  C.max_lot_size,
  C.fixed_lot_size,
  C.max_stock_level,
  C.lot_ordering_cost,
  C.mrp_time_fence,
  C.ext_procure_time,
  C.internal_mfg_time,
  C.max_storage_days,
  C.mrp_profile_code,
  C.mrp_type_code,
  C.mrp_grp_code,
  C.lot_size_code,
  C.backflush_ind,
  C.qa_inspect_ind,
  C.repetitive_mfg_ind,
  C.bulk_item_ind,
  C.forecast_period,
  C.mfg_uom_code,
  C.issue_uom_code,
  C.manufacturing_place,
  C.loading_type_code,
  C.int_store_loc_code,
  C.ext_store_loc_code,
  C.active_flg,
  C.created_by_wid,
  C.changed_by_wid,
  C.created_on_dt,
  C.changed_on_dt,
  C.aux1_changed_on_dt,
  C.aux2_changed_on_dt,
  C.aux3_changed_on_dt,
  C.aux4_changed_on_dt,
  C.src_eff_from_dt,
  C.src_eff_to_dt,
  C.effective_from_dt,
  C.delete_flg,
  C.datasource_num_id,
  C.integration_id,
  C.tenant_id,
  C.x_custom,
  C.inv_prod_cat1,
  C.inv_prod_cat2,
  C.inv_prod_cat3,
  C.inv_prod_cat4,
  C.inv_prod_cat5,
  C.inv_prod_cat6,
  C.inv_prod_cat7,
  C.inv_prod_cat8,
  C.inv_prod_cat9,
  C.inv_prod_cat10,
  C.inv_prod_cat1_wid,
  C.inv_prod_cat2_wid,
  C.inv_prod_cat3_wid,
  C.inv_prod_cat4_wid,
  C.inv_prod_cat5_wid,
  C.inv_prod_cat6_wid,
  C.inv_prod_cat7_wid,
  C.inv_prod_cat8_wid,
  C.inv_prod_cat9_wid,
  C.inv_prod_cat10_wid,
  C.invoiceable_item_flag,
  C.invoice_enabled_flag,
  C.primary_uom_code,
  C.c_primary_uom_code,
  C.unspsc_code,
  C.unspsc_inv_prod_cat_wid,
  C.commodity_name,
  C.commodity_uom_name,
  C.ext_store_loc_name,
  C.int_store_loc_name,
  C.issue_uom_name,
  C.loading_type_name,
  C.lot_size_name,
  C.mfg_uom_name,
  C.mrp_grp_name,
  C.mrp_profile_name,
  C.mrp_type_name,
  C.planner_name,
  C.primary_uom_name,
  C.procurement_type_name,
  C.profit_center_name,
  C.spc_proc_type_name,
  C.status_code,
  C.w_status_code,
  C.product_type_code,
  C.make_buy_ind,
  C.fixed_lead_time,
  C.variable_lead_time,
  C.cumulative_total_lead_time,
  C.postprocessing_lead_time,
  C.preprocessing_lead_time,
  C.process_quality_enabled_flg,
  C.x_price_sequence,
  C.x_organization_name,
  C.x_product_desc,
  C.x_uom_desc,
  C.x_inv_item_flg,
  C.x_stock_item_flg,
  C.x_trans_flg,
  C.x_rev_flg,
  C.x_cost_flg,
  C.x_gcoa_acct,
  C.x_gcoa_prod,
  C.x_tax_cat,
  C.organization_id,
  C.x_gcoa_loc_acct,
  'Y',
  to_timestamp('3714-01-01 00:00:00', 'yyyy-MM-dd HH:mm:ss'),
  CASE
    WHEN T.src_eff_from_dt IS NOT NULL
    AND (
      T.changed_on_dt = C.changed_on_dt
      OR (T.changed_on_dt IS NULL AND C.changed_on_dt IS NULL)
    )
    AND (
      T.aux1_changed_on_dt = C.aux1_changed_on_dt
      OR (T.aux1_changed_on_dt IS NULL AND C.aux1_changed_on_dt IS NULL)
    )
    AND (
      T.aux2_changed_on_dt = C.aux2_changed_on_dt
      OR (T.aux2_changed_on_dt IS NULL AND C.aux2_changed_on_dt IS NULL)
    )
    AND (
      T.aux3_changed_on_dt = C.aux3_changed_on_dt
      OR (T.aux3_changed_on_dt IS NULL AND C.aux3_changed_on_dt IS NULL)
    )
    AND (
      T.aux4_changed_on_dt = C.aux4_changed_on_dt
      OR (T.aux4_changed_on_dt IS NULL AND C.aux4_changed_on_dt IS NULL)
    )
    THEN 'N'
    WHEN T.src_eff_from_dt IS NOT NULL
    THEN 'U'
    ELSE 'I'
  END AS ind_update
FROM
  (
    SELECT
      COALESCE(INLINE_VIEW.scd1_wid_1, 0) AS product_wid,
      COALESCE(INLINE_VIEW.scd1_wid, 0) AS inventory_org_wid,
      COALESCE(INLINE_VIEW.row_wid, 0) AS plant_loc_wid,
      INLINE_VIEW.product_num AS product_num,
      INLINE_VIEW.abc_ind AS abc_ind,
      COALESCE(INLINE_VIEW.planner_code, '__NOT_APPLICABLE__') AS planner_code,
      COALESCE(INLINE_VIEW.procurement_type_code, '__NOT_APPLICABLE__') AS procurement_type_code,
      COALESCE(INLINE_VIEW.spc_proc_type_code, '__NOT_APPLICABLE__') AS spc_proc_type_code,
      COALESCE(INLINE_VIEW.buyer_code, '__NOT_APPLICABLE__') AS buyer_code,
      INLINE_VIEW.buyer_name AS buyer_name,
      COALESCE(INLINE_VIEW.commodity_code, '__NOT_APPLICABLE__') AS commodity_code,
      COALESCE(INLINE_VIEW.commodity_uom_code, '__NOT_APPLICABLE__') AS commodity_uom_code,
      INLINE_VIEW.profit_center_num AS profit_center_num,
      INLINE_VIEW.reorder_point AS reorder_point,
      INLINE_VIEW.safety_stock_level AS safety_stock_level,
      INLINE_VIEW.min_lot_size AS min_lot_size,
      INLINE_VIEW.max_lot_size AS max_lot_size,
      INLINE_VIEW.fixed_lot_size AS fixed_lot_size,
      INLINE_VIEW.max_stock_level AS max_stock_level,
      INLINE_VIEW.lot_ordering_cost AS lot_ordering_cost,
      INLINE_VIEW.mrp_time_fence AS mrp_time_fence,
      INLINE_VIEW.ext_procure_time AS ext_procure_time,
      INLINE_VIEW.internal_mfg_time AS internal_mfg_time,
      INLINE_VIEW.max_storage_days AS max_storage_days,
      COALESCE(INLINE_VIEW.mrp_profile_code, '__NOT_APPLICABLE__') AS mrp_profile_code,
      COALESCE(INLINE_VIEW.mrp_type_code, '__NOT_APPLICABLE__') AS mrp_type_code,
      COALESCE(INLINE_VIEW.mrp_grp_code, '__NOT_APPLICABLE__') AS mrp_grp_code,
      COALESCE(INLINE_VIEW.lot_size_code, '__NOT_APPLICABLE__') AS lot_size_code,
      INLINE_VIEW.backflush_ind AS backflush_ind,
      INLINE_VIEW.qa_inspect_ind AS qa_inspect_ind,
      INLINE_VIEW.repetitive_mfg_ind AS repetitive_mfg_ind,
      INLINE_VIEW.bulk_item_ind AS bulk_item_ind,
      INLINE_VIEW.forecast_period AS forecast_period,
      COALESCE(INLINE_VIEW.mfg_uom_code, '__NOT_APPLICABLE__') AS mfg_uom_code,
      COALESCE(INLINE_VIEW.issue_uom_code, '__NOT_APPLICABLE__') AS issue_uom_code,
      INLINE_VIEW.manufacturing_place AS manufacturing_place,
      COALESCE(INLINE_VIEW.loading_type_code, '__NOT_APPLICABLE__') AS loading_type_code,
      COALESCE(INLINE_VIEW.int_store_loc_code, '__NOT_APPLICABLE__') AS int_store_loc_code,
      COALESCE(INLINE_VIEW.ext_store_loc_code, '__NOT_APPLICABLE__') AS ext_store_loc_code,
      INLINE_VIEW.active_flg AS active_flg,
      COALESCE(LKP_W_USER_D_LKP_W_USER_D_CR_1.row_wid, 0) AS created_by_wid,
      COALESCE(INLINE_VIEW.row_wid_1, 0) AS changed_by_wid,
      INLINE_VIEW.created_on_dt AS created_on_dt,
      INLINE_VIEW.changed_on_dt AS changed_on_dt,
      INLINE_VIEW.aux1_changed_on_dt AS aux1_changed_on_dt,
      INLINE_VIEW.aux2_changed_on_dt AS aux2_changed_on_dt,
      INLINE_VIEW.aux3_changed_on_dt AS aux3_changed_on_dt,
      INLINE_VIEW.aux4_changed_on_dt AS aux4_changed_on_dt,
      INLINE_VIEW.src_eff_from_dt AS src_eff_from_dt,
      INLINE_VIEW.src_eff_to_dt AS src_eff_to_dt,
      COALESCE(INLINE_VIEW.src_eff_from_dt, to_timestamp(substring((SELECT BIAPPS_LOW_DATE FROM v_bi_apps_params), 1, 19), 'yyyy-MM-dd HH:mm:ss')) AS effective_from_dt,
      (CASE
        WHEN INLINE_VIEW.delete_flg = 'Y'
        THEN 'Y'
        ELSE 'N'
      END) AS delete_flg,
      INLINE_VIEW.datasource_num_id AS datasource_num_id,
      INLINE_VIEW.integration_id AS integration_id,
      INLINE_VIEW.tenant_id AS tenant_id,
      INLINE_VIEW.x_custom AS x_custom,
      INLINE_VIEW.inv_prod_cat1 AS inv_prod_cat1,
      INLINE_VIEW.inv_prod_cat2 AS inv_prod_cat2,
      INLINE_VIEW.inv_prod_cat3 AS inv_prod_cat3,
      INLINE_VIEW.inv_prod_cat4 AS inv_prod_cat4,
      INLINE_VIEW.inv_prod_cat5 AS inv_prod_cat5,
      INLINE_VIEW.inv_prod_cat6 AS inv_prod_cat6,
      INLINE_VIEW.inv_prod_cat7 AS inv_prod_cat7,
      INLINE_VIEW.inv_prod_cat8 AS inv_prod_cat8,
      INLINE_VIEW.inv_prod_cat9 AS inv_prod_cat9,
      INLINE_VIEW.inv_prod_cat10 AS inv_prod_cat10,
      (CASE
        WHEN INLINE_VIEW.inv_prod_cat1 IS NULL
        THEN 0
        ELSE COALESCE(INLINE_VIEW.inv_prod_cat1_row_wid, 0)
      END) AS inv_prod_cat1_wid,
      (CASE
        WHEN INLINE_VIEW.inv_prod_cat2 IS NULL
        THEN 0
        ELSE COALESCE(INLINE_VIEW.inv_prod_cat2_row_wid, 0)
      END) AS inv_prod_cat2_wid,
      (CASE
        WHEN INLINE_VIEW.inv_prod_cat3 IS NULL
        THEN 0
        ELSE COALESCE(INLINE_VIEW.inv_prod_cat3_row_wid, 0)
      END) AS inv_prod_cat3_wid,
      (CASE
        WHEN INLINE_VIEW.inv_prod_cat4 IS NULL
        THEN 0
        ELSE COALESCE(INLINE_VIEW.inv_prod_cat4_row_wid, 0)
      END) AS inv_prod_cat4_wid,
      (CASE
        WHEN INLINE_VIEW.inv_prod_cat5 IS NULL
        THEN 0
        ELSE COALESCE(INLINE_VIEW.inv_prod_cat5_row_wid, 0)
      END) AS inv_prod_cat5_wid,
      (CASE
        WHEN INLINE_VIEW.inv_prod_cat6 IS NULL
        THEN 0
        ELSE COALESCE(INLINE_VIEW.inv_prod_cat6_row_wid, 0)
      END) AS inv_prod_cat6_wid,
      (CASE
        WHEN INLINE_VIEW.inv_prod_cat7 IS NULL
        THEN 0
        ELSE COALESCE(INLINE_VIEW.inv_prod_cat7_row_wid, 0)
      END) AS inv_prod_cat7_wid,
      (CASE
        WHEN INLINE_VIEW.inv_prod_cat8 IS NULL
        THEN 0
        ELSE COALESCE(INLINE_VIEW.inv_prod_cat8_row_wid, 0)
      END) AS inv_prod_cat8_wid,
      (CASE
        WHEN INLINE_VIEW.inv_prod_cat9 IS NULL
        THEN 0
        ELSE COALESCE(INLINE_VIEW.inv_prod_cat9_row_wid, 0)
      END) AS inv_prod_cat9_wid,
      (CASE
        WHEN INLINE_VIEW.inv_prod_cat10 IS NULL
        THEN 0
        ELSE COALESCE(INLINE_VIEW.inv_prod_cat10_row_wid, 0)
      END) AS inv_prod_cat10_wid,
      COALESCE(INLINE_VIEW.invoiceable_item_flag, 'N') AS invoiceable_item_flag,
      COALESCE(INLINE_VIEW.invoice_enabled_flag, 'N') AS invoice_enabled_flag,
      COALESCE(INLINE_VIEW.primary_uom_code, '__NOT_APPLICABLE__') AS primary_uom_code,
      COALESCE(
        (
          SELECT
            W_DOMAIN_MEMBER_MAP_G.trg_domain_member_code
          FROM
            workspace.prxbi_dw.w_domain_member_map_g AS W_DOMAIN_MEMBER_MAP_G
          WHERE
            W_DOMAIN_MEMBER_MAP_G.src_domain_code = (SELECT BIAPPS_SOURCE_CODE FROM v_bi_apps_params)
            AND W_DOMAIN_MEMBER_MAP_G.src_domain_member_code = COALESCE(INLINE_VIEW.primary_uom_code, '__UNASSIGNED__')
            AND W_DOMAIN_MEMBER_MAP_G.src_datasource_num_id IN (INLINE_VIEW.datasource_num_id, 999)
            AND W_DOMAIN_MEMBER_MAP_G.trg_domain_code = (SELECT BIAPPS_TARGET_CODE FROM v_bi_apps_params)
        ),
        (
          SELECT
            W_DOMAIN_MEMBER_MAP_G.trg_domain_member_code
          FROM
            workspace.prxbi_dw.w_domain_member_map_g AS W_DOMAIN_MEMBER_MAP_G
          WHERE
            W_DOMAIN_MEMBER_MAP_G.src_domain_code = (SELECT BIAPPS_SOURCE_CODE FROM v_bi_apps_params)
            AND W_DOMAIN_MEMBER_MAP_G.src_domain_member_code = '__ANY__'
            AND W_DOMAIN_MEMBER_MAP_G.src_datasource_num_id IN (INLINE_VIEW.datasource_num_id, 999)
            AND W_DOMAIN_MEMBER_MAP_G.trg_domain_code = (SELECT BIAPPS_TARGET_CODE FROM v_bi_apps_params)
        ),
        (CASE
          WHEN INLINE_VIEW.primary_uom_code IS NULL
          THEN COALESCE(
            (
              SELECT
                W_DOMAIN_MEMBER_G.domain_member_code
              FROM
                workspace.prxbi_dw.w_domain_member_g AS W_DOMAIN_MEMBER_G
              WHERE
                W_DOMAIN_MEMBER_G.domain_member_code = '__UNASSIGNED__'
                AND domain_code = (SELECT BIAPPS_TARGET_CODE FROM v_bi_apps_params)
            ),
            '__ERROR__'
          )
          ELSE (
            CASE
              WHEN (SELECT BIAPPS_TARGET_CODE FROM v_bi_apps_params) = 'W_LANGUAGE'
              THEN '_ERR'
              ELSE '__ERROR__'
            END
          )
        END)
      ) AS c_primary_uom_code,
      COALESCE(INLINE_VIEW.unspsc_code, '__NOT_APPLICABLE__') AS unspsc_code,
      COALESCE(INLINE_VIEW.inv_prod_cat_unspsc_row_wid, 0) AS unspsc_inv_prod_cat_wid,
      INLINE_VIEW.commodity_name AS commodity_name,
      INLINE_VIEW.commodity_uom_name AS commodity_uom_name,
      INLINE_VIEW.ext_store_loc_name AS ext_store_loc_name,
      INLINE_VIEW.int_store_loc_name AS int_store_loc_name,
      INLINE_VIEW.issue_uom_name AS issue_uom_name,
      INLINE_VIEW.loading_type_name AS loading_type_name,
      INLINE_VIEW.lot_size_name AS lot_size_name,
      INLINE_VIEW.mfg_uom_name AS mfg_uom_name,
      INLINE_VIEW.mrp_grp_name AS mrp_grp_name,
      INLINE_VIEW.mrp_profile_name AS mrp_profile_name,
      INLINE_VIEW.mrp_type_name AS mrp_type_name,
      INLINE_VIEW.planner_name AS planner_name,
      INLINE_VIEW.primary_uom_name AS primary_uom_name,
      INLINE_VIEW.procurement_type_name AS procurement_type_name,
      INLINE_VIEW.profit_center_name AS profit_center_name,
      INLINE_VIEW.spc_proc_type_name AS spc_proc_type_name,
      INLINE_VIEW.status_code AS status_code,
      INLINE_VIEW.w_status_code AS w_status_code,
      INLINE_VIEW.product_type_code AS product_type_code,
      INLINE_VIEW.make_buy_ind AS make_buy_ind,
      INLINE_VIEW.fixed_lead_time AS fixed_lead_time,
      INLINE_VIEW.variable_lead_time AS variable_lead_time,
      INLINE_VIEW.cumulative_total_lead_time AS cumulative_total_lead_time,
      INLINE_VIEW.preprocessing_lead_time AS postprocessing_lead_time,
      INLINE_VIEW.preprocessing_lead_time AS preprocessing_lead_time,
      INLINE_VIEW.process_quality_enabled_flg AS process_quality_enabled_flg,
      INLINE_VIEW.x_price_sequence AS x_price_sequence,
      INLINE_VIEW.x_organization_name AS x_organization_name,
      INLINE_VIEW.x_product_desc AS x_product_desc,
      INLINE_VIEW.x_uom_desc AS x_uom_desc,
      INLINE_VIEW.x_inv_item_flg AS x_inv_item_flg,
      INLINE_VIEW.x_stock_item_flg AS x_stock_item_flg,
      INLINE_VIEW.x_trans_flg AS x_trans_flg,
      INLINE_VIEW.x_rev_flg AS x_rev_flg,
      INLINE_VIEW.x_cost_flg AS x_cost_flg,
      INLINE_VIEW.x_gcoa_acct AS x_gcoa_acct,
      INLINE_VIEW.x_gcoa_prod AS x_gcoa_prod,
      INLINE_VIEW.x_tax_cat AS x_tax_cat,
      INLINE_VIEW.inventory_org_id AS organization_id,
      INLINE_VIEW.x_gcoa_loc_acct AS x_gcoa_loc_acct
    FROM
      (
        SELECT
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mrp_type_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_custom,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.src_eff_to_dt,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.ext_procure_time,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.commodity_uom_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat10,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.invoice_enabled_flag,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mrp_profile_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.ext_store_loc_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.aux3_changed_on_dt,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.min_lot_size,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.reorder_point,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.bulk_item_ind,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.spc_proc_type_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.commodity_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.repetitive_mfg_ind,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.loading_type_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.profit_center_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_by_id,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.buyer_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.int_store_loc_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.commodity_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.planner_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.max_storage_days,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.planner_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.max_lot_size,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mrp_grp_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.lot_ordering_cost,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.internal_mfg_time,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.manufacturing_place,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.procurement_type_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.plant_loc_id,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.changed_by_id,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.backflush_ind,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.aux2_changed_on_dt,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.spc_proc_type_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.lot_size_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.changed_on_dt,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.product_id,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.primary_uom_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.product_num,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.forecast_period,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.aux1_changed_on_dt,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.buyer_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.fixed_lot_size,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mrp_grp_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.procurement_type_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.primary_uom_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.max_stock_level,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.safety_stock_level,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.int_store_loc_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.ext_store_loc_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.issue_uom_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.abc_ind,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.profit_center_num,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_on_dt,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.unspsc_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat1,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat3,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat2,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.commodity_uom_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat5,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat4,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mfg_uom_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat7,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat6,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat9,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat8,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.aux4_changed_on_dt,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.lot_size_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.tenant_id,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.invoiceable_item_flag,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inventory_org_id,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.src_eff_from_dt,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.integration_id,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.issue_uom_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.delete_flg,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mrp_type_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.active_flg,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mrp_time_fence,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.qa_inspect_ind,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mfg_uom_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.loading_type_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.mrp_profile_name,
          W_PROD_CAT_DH1_SQ_W_INVENTORY_.row_wid AS inv_prod_cat1_row_wid,
          W_PROD_CAT_DH2_SQ_W_INVENTORY_.row_wid AS inv_prod_cat2_row_wid,
          W_PROD_CAT_DH3_SQ_W_INVENTORY_.row_wid AS inv_prod_cat3_row_wid,
          W_PROD_CAT_DH4_SQ_W_INVENTORY_.row_wid AS inv_prod_cat4_row_wid,
          W_PROD_CAT_DH5_SQ_W_INVENTORY_.row_wid AS inv_prod_cat5_row_wid,
          W_PROD_CAT_DH6_SQ_W_INVENTORY_.row_wid AS inv_prod_cat6_row_wid,
          W_PROD_CAT_DH7_SQ_W_INVENTORY_.row_wid AS inv_prod_cat7_row_wid,
          W_PROD_CAT_DH8_SQ_W_INVENTORY_.row_wid AS inv_prod_cat8_row_wid,
          W_PROD_CAT_DH_UNSPSC_SQ_W_INVE.row_wid AS inv_prod_cat_unspsc_row_wid,
          W_PROD_CAT_DH9_SQ_W_INVENTORY_.row_wid AS inv_prod_cat9_row_wid,
          W_PROD_CAT_DH10_SQ_W_INVENTORY.row_wid AS inv_prod_cat10_row_wid,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.cumulative_total_lead_time,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.fixed_lead_time,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.w_status_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.preprocessing_lead_time,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.status_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.variable_lead_time,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.make_buy_ind,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.process_quality_enabled_flg,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.product_type_code,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_price_sequence,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_organization_name,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_product_desc,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_uom_desc,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_inv_item_flg,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_stock_item_flg,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_trans_flg,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_rev_flg,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_cost_flg,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_gcoa_acct,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_gcoa_prod,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_tax_cat,
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.x_gcoa_loc_acct,
          LKP_W_BUSN_LOCATION_D_LKP_W_BU.datasource_num_id AS datasource_num_id_1,
          LKP_W_BUSN_LOCATION_D_LKP_W_BU.row_wid AS row_wid,
          LKP_W_BUSN_LOCATION_D_LKP_W_BU.integration_id AS integration_id_1,
          LKP_W_BUSN_LOCATION_D_LKP_W_BU.effective_to_dt AS effective_to_dt,
          LKP_W_BUSN_LOCATION_D_LKP_W_BU.effective_from_dt AS effective_from_dt,
          LKP_W_INT_ORG_D_INVENTORY.effective_from_dt AS effective_from_dt_1,
          LKP_W_INT_ORG_D_INVENTORY.effective_to_dt AS effective_to_dt_1,
          LKP_W_INT_ORG_D_INVENTORY.datasource_num_id AS datasource_num_id_2,
          LKP_W_INT_ORG_D_INVENTORY.integration_id AS integration_id_2,
          LKP_W_INT_ORG_D_INVENTORY.scd1_wid AS scd1_wid,
          LKP_W_PRODUCT_D_PRODUCT_WID.effective_from_dt AS effective_from_dt_2,
          LKP_W_PRODUCT_D_PRODUCT_WID.effective_to_dt AS effective_to_dt_2,
          LKP_W_PRODUCT_D_PRODUCT_WID.datasource_num_id AS datasource_num_id_3,
          LKP_W_PRODUCT_D_PRODUCT_WID.integration_id AS integration_id_3,
          LKP_W_PRODUCT_D_PRODUCT_WID.scd1_wid AS scd1_wid_1,
          LKP_W_USER_D_LKP_W_USER_D_CHAN.datasource_num_id AS datasource_num_id_4,
          LKP_W_USER_D_LKP_W_USER_D_CHAN.row_wid AS row_wid_1,
          LKP_W_USER_D_LKP_W_USER_D_CHAN.integration_id AS integration_id_4,
          LKP_W_USER_D_LKP_W_USER_D_CHAN.effective_to_dt AS effective_to_dt_3,
          LKP_W_USER_D_LKP_W_USER_D_CHAN.effective_from_dt AS effective_from_dt_3
        FROM
          (
            (
              (
                (
                  (
                    (
                      (
                        (
                          (
                            (
                              workspace.prxbi_dw.w_inventory_product_ds AS SQ_W_INVENTORY_PRODUCT_DS_SQ_W
                            LEFT OUTER JOIN
                              workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH1_SQ_W_INVENTORY_
                            ON
                              SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat1 = W_PROD_CAT_DH1_SQ_W_INVENTORY_.integration_id
                              AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = W_PROD_CAT_DH1_SQ_W_INVENTORY_.datasource_num_id
                            )
                          LEFT OUTER JOIN
                            workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH2_SQ_W_INVENTORY_
                          ON
                            SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat2 = W_PROD_CAT_DH2_SQ_W_INVENTORY_.integration_id
                            AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = W_PROD_CAT_DH2_SQ_W_INVENTORY_.datasource_num_id
                          )
                        LEFT OUTER JOIN
                          workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH3_SQ_W_INVENTORY_
                        ON
                          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat3 = W_PROD_CAT_DH3_SQ_W_INVENTORY_.integration_id
                          AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = W_PROD_CAT_DH3_SQ_W_INVENTORY_.datasource_num_id
                        )
                      LEFT OUTER JOIN
                        workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH4_SQ_W_INVENTORY_
                      ON
                        SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat4 = W_PROD_CAT_DH4_SQ_W_INVENTORY_.integration_id
                        AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = W_PROD_CAT_DH4_SQ_W_INVENTORY_.datasource_num_id
                      )
                    LEFT OUTER JOIN
                      workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH5_SQ_W_INVENTORY_
                    ON
                      SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat5 = W_PROD_CAT_DH5_SQ_W_INVENTORY_.integration_id
                      AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = W_PROD_CAT_DH5_SQ_W_INVENTORY_.datasource_num_id
                    )
                  LEFT OUTER JOIN
                    workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH6_SQ_W_INVENTORY_
                  ON
                    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat6 = W_PROD_CAT_DH6_SQ_W_INVENTORY_.integration_id
                    AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = W_PROD_CAT_DH6_SQ_W_INVENTORY_.datasource_num_id
                  )
                LEFT OUTER JOIN
                  workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH7_SQ_W_INVENTORY_
                ON
                  SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat7 = W_PROD_CAT_DH7_SQ_W_INVENTORY_.integration_id
                  AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = W_PROD_CAT_DH7_SQ_W_INVENTORY_.datasource_num_id
                )
              LEFT OUTER JOIN
                workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH8_SQ_W_INVENTORY_
              ON
                SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat8 = W_PROD_CAT_DH8_SQ_W_INVENTORY_.integration_id
                AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = W_PROD_CAT_DH8_SQ_W_INVENTORY_.datasource_num_id
              )
            LEFT OUTER JOIN
              workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH9_SQ_W_INVENTORY_
            ON
              SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat9 = W_PROD_CAT_DH9_SQ_W_INVENTORY_.integration_id
              AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = W_PROD_CAT_DH9_SQ_W_INVENTORY_.datasource_num_id
            )
          LEFT OUTER JOIN
            workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH10_SQ_W_INVENTORY
          ON
            SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inv_prod_cat10 = W_PROD_CAT_DH10_SQ_W_INVENTORY.integration_id
            AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = W_PROD_CAT_DH10_SQ_W_INVENTORY.datasource_num_id
          )
        LEFT OUTER JOIN
          workspace.prxbi_dw.w_prod_cat_dh AS W_PROD_CAT_DH_UNSPSC_SQ_W_INVE
        ON
          SQ_W_INVENTORY_PRODUCT_DS_SQ_W.unspsc_code = W_PROD_CAT_DH_UNSPSC_SQ_W_INVE.integration_id
          AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = W_PROD_CAT_DH_UNSPSC_SQ_W_INVE.datasource_num_id
        WHERE
          (1 = 1)
      ) AS SQ_W_INVENTORY_PRODUCT_DS_SQ_W
    LEFT OUTER JOIN
      (
        SELECT
          W_BUSN_LOCATION_D_LKP_W_BUSN_L.datasource_num_id,
          W_BUSN_LOCATION_D_LKP_W_BUSN_L.row_wid,
          W_BUSN_LOCATION_D_LKP_W_BUSN_L.integration_id,
          W_BUSN_LOCATION_D_LKP_W_BUSN_L.effective_to_dt,
          W_BUSN_LOCATION_D_LKP_W_BUSN_L.effective_from_dt
        FROM
          workspace.prxbi_dw.w_busn_location_d AS W_BUSN_LOCATION_D_LKP_W_BUSN_L
        WHERE
          (1 = 1)
      ) AS LKP_W_BUSN_LOCATION_D_LKP_W_BU
    ON
      SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = LKP_W_BUSN_LOCATION_D_LKP_W_BU.datasource_num_id
      AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.plant_loc_id = LKP_W_BUSN_LOCATION_D_LKP_W_BU.integration_id
      AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_on_dt >= LKP_W_BUSN_LOCATION_D_LKP_W_BU.effective_from_dt
      AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_on_dt < LKP_W_BUSN_LOCATION_D_LKP_W_BU.effective_to_dt
    )
  LEFT OUTER JOIN
    workspace.prxbi_dw.w_int_org_d AS LKP_W_INT_ORG_D_INVENTORY
  ON
    SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = LKP_W_INT_ORG_D_INVENTORY.datasource_num_id
    AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.inventory_org_id = LKP_W_INT_ORG_D_INVENTORY.integration_id
    AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_on_dt >= LKP_W_INT_ORG_D_INVENTORY.effective_from_dt
    AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_on_dt < LKP_W_INT_ORG_D_INVENTORY.effective_to_dt
  )
LEFT OUTER JOIN
  workspace.prxbi_dw.w_product_d AS LKP_W_PRODUCT_D_PRODUCT_WID
ON
  SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = LKP_W_PRODUCT_D_PRODUCT_WID.datasource_num_id
  AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.product_id = LKP_W_PRODUCT_D_PRODUCT_WID.integration_id
  AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_on_dt >= LKP_W_PRODUCT_D_PRODUCT_WID.effective_from_dt
  AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.created_on_dt < LKP_W_PRODUCT_D_PRODUCT_WID.effective_to_dt
)
LEFT OUTER JOIN
  (
    SELECT
      W_USER_D_LKP_W_USER_D_CHANGED_.datasource_num_id,
      W_USER_D_LKP_W_USER_D_CHANGED_.row_wid,
      W_USER_D_LKP_W_USER_D_CHANGED_.integration_id,
      W_USER_D_LKP_W_USER_D_CHANGED_.effective_to_dt,
      W_USER_D_LKP_W_USER_D_CHANGED_.effective_from_dt
    FROM
      workspace.prxbi_dw.w_user_d AS W_USER_D_LKP_W_USER_D_CHANGED_
    WHERE
      (1 = 1)
      AND (W_USER_D_LKP_W_USER_D_CHANGED_.delete_flg = 'N')
  ) AS LKP_W_USER_D_LKP_W_USER_D_CHAN
ON
  SQ_W_INVENTORY_PRODUCT_DS_SQ_W.datasource_num_id = LKP_W_USER_D_LKP_W_USER_D_CHAN.datasource_num_id
  AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.changed_by_id = LKP_W_USER_D_LKP_W_USER_D_CHAN.integration_id
  AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.changed_on_dt >= LKP_W_USER_D_LKP_W_USER_D_CHAN.effective_from_dt
  AND SQ_W_INVENTORY_PRODUCT_DS_SQ_W.changed_on_dt < LKP_W_USER_D_LKP_W_USER_D_CHAN.effective_to_dt
) AS INLINE_VIEW
LEFT OUTER JOIN
  (
    SELECT
      LKP_W_USER_D_LKP_W_USER_D_CREA.datasource_num_id,
      LKP_W_USER_D_LKP_W_USER_D_CREA.row_wid,
      LKP_W_USER_D_LKP_W_USER_D_CREA.integration_id,
      LKP_W_USER_D_LKP_W_USER_D_CREA.effective_to_dt,
      LKP_W_USER_D_LKP_W_USER_D_CREA.effective_from_dt
    FROM
      (
        SELECT
          W_USER_D_LKP_W_USER_D_CREATED_.datasource_num_id,
          W_USER_D_LKP_W_USER_D_CREATED_.row_wid,
          W_USER_D_LKP_W_USER_D_CREATED_.integration_id,
          W_USER_D_LKP_W_USER_D_CREATED_.effective_to_dt,
          W_USER_D_LKP_W_USER_D_CREATED_.effective_from_dt
        FROM
          workspace.prxbi_dw.w_user_d AS W_USER_D_LKP_W_USER_D_CREATED_
        WHERE
          (1 = 1)
          AND (W_USER_D_LKP_W_USER_D_CREATED_.delete_flg = 'N')
      ) AS LKP_W_USER_D_LKP_W_USER_D_CREA
  ) AS LKP_W_USER_D_LKP_W_USER_D_CR_1
ON
  INLINE_VIEW.datasource_num_id = LKP_W_USER_D_LKP_W_USER_D_CR_1.datasource_num_id
  AND INLINE_VIEW.created_by_id = LKP_W_USER_D_LKP_W_USER_D_CR_1.integration_id
  AND INLINE_VIEW.created_on_dt >= LKP_W_USER_D_LKP_W_USER_D_CR_1.effective_from_dt
  AND INLINE_VIEW.created_on_dt < LKP_W_USER_D_LKP_W_USER_D_CR_1.effective_to_dt
WHERE
  (1 = 1)
) AS C
LEFT OUTER JOIN
  workspace.prxbi_dw.w_inventory_product_d AS T
ON
  C.src_eff_from_dt = T.src_eff_from_dt
  AND C.datasource_num_id = T.datasource_num_id
  AND C.integration_id = T.integration_id;

In [ ]:
%sql
-- SCEN_TASK_NO {150}: Optimize flow table
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.prxbi_dw.i_inventory_product_d_flow
ZORDER BY (src_eff_from_dt, datasource_num_id, integration_id, ind_update);

## Merge into Target Table (SCEN_TASK_NO {330} & {350})

In [ ]:
%sql
MERGE INTO workspace.prxbi_dw.w_inventory_product_d AS T
USING workspace.prxbi_dw.i_inventory_product_d_flow AS S
ON
  T.src_eff_from_dt = S.src_eff_from_dt
  AND T.datasource_num_id = S.datasource_num_id
  AND T.integration_id = S.integration_id
WHEN MATCHED AND S.ind_update = 'U' THEN
  UPDATE SET
    T.product_wid = S.product_wid,
    T.inventory_org_wid = S.inventory_org_wid,
    T.plant_loc_wid = S.plant_loc_wid,
    T.product_num = S.product_num,
    T.abc_ind = S.abc_ind,
    T.planner_code = S.planner_code,
    T.procurement_type_code = S.procurement_type_code,
    T.spc_proc_type_code = S.spc_proc_type_code,
    T.buyer_code = S.buyer_code,
    T.buyer_name = S.buyer_name,
    T.commodity_code = S.commodity_code,
    T.commodity_uom_code = S.commodity_uom_code,
    T.profit_center_num = S.profit_center_num,
    T.reorder_point = S.reorder_point,
    T.safety_stock_level = S.safety_stock_level,
    T.min_lot_size = S.min_lot_size,
    T.max_lot_size = S.max_lot_size,
    T.fixed_lot_size = S.fixed_lot_size,
    T.max_stock_level = S.max_stock_level,
    T.lot_ordering_cost = S.lot_ordering_cost,
    T.mrp_time_fence = S.mrp_time_fence,
    T.ext_procure_time = S.ext_procure_time,
    T.internal_mfg_time = S.internal_mfg_time,
    T.max_storage_days = S.max_storage_days,
    T.mrp_profile_code = S.mrp_profile_code,
    T.mrp_type_code = S.mrp_type_code,
    T.mrp_grp_code = S.mrp_grp_code,
    T.lot_size_code = S.lot_size_code,
    T.backflush_ind = S.backflush_ind,
    T.qa_inspect_ind = S.qa_inspect_ind,
    T.repetitive_mfg_ind = S.repetitive_mfg_ind,
    T.bulk_item_ind = S.bulk_item_ind,
    T.forecast_period = S.forecast_period,
    T.mfg_uom_code = S.mfg_uom_code,
    T.issue_uom_code = S.issue_uom_code,
    T.manufacturing_place = S.manufacturing_place,
    T.loading_type_code = S.loading_type_code,
    T.int_store_loc_code = S.int_store_loc_code,
    T.ext_store_loc_code = S.ext_store_loc_code,
    T.active_flg = S.active_flg,
    T.created_by_wid = S.created_by_wid,
    T.changed_by_wid = S.changed_by_wid,
    T.created_on_dt = S.created_on_dt,
    T.changed_on_dt = S.changed_on_dt,
    T.aux1_changed_on_dt = S.aux1_changed_on_dt,
    T.aux2_changed_on_dt = S.aux2_changed_on_dt,
    T.aux3_changed_on_dt = S.aux3_changed_on_dt,
    T.aux4_changed_on_dt = S.aux4_changed_on_dt,
    T.src_eff_to_dt = S.src_eff_to_dt,
    T.effective_from_dt = S.effective_from_dt,
    T.delete_flg = S.delete_flg,
    T.tenant_id = S.tenant_id,
    T.x_custom = S.x_custom,
    T.inv_prod_cat1 = S.inv_prod_cat1,
    T.inv_prod_cat2 = S.inv_prod_cat2,
    T.inv_prod_cat3 = S.inv_prod_cat3,
    T.inv_prod_cat4 = S.inv_prod_cat4,
    T.inv_prod_cat5 = S.inv_prod_cat5,
    T.inv_prod_cat6 = S.inv_prod_cat6,
    T.inv_prod_cat7 = S.inv_prod_cat7,
    T.inv_prod_cat8 = S.inv_prod_cat8,
    T.inv_prod_cat9 = S.inv_prod_cat9,
    T.inv_prod_cat10 = S.inv_prod_cat10,
    T.inv_prod_cat1_wid = S.inv_prod_cat1_wid,
    T.inv_prod_cat2_wid = S.inv_prod_cat2_wid,
    T.inv_prod_cat3_wid = S.inv_prod_cat3_wid,
    T.inv_prod_cat4_wid = S.inv_prod_cat4_wid,
    T.inv_prod_cat5_wid = S.inv_prod_cat5_wid,
    T.inv_prod_cat6_wid = S.inv_prod_cat6_wid,
    T.inv_prod_cat7_wid = S.inv_prod_cat7_wid,
    T.inv_prod_cat8_wid = S.inv_prod_cat8_wid,
    T.inv_prod_cat9_wid = S.inv_prod_cat9_wid,
    T.inv_prod_cat10_wid = S.inv_prod_cat10_wid,
    T.invoiceable_item_flag = S.invoiceable_item_flag,
    T.invoice_enabled_flag = S.invoice_enabled_flag,
    T.primary_uom_code = S.primary_uom_code,
    T.c_primary_uom_code = S.c_primary_uom_code,
    T.unspsc_code = S.unspsc_code,
    T.unspsc_inv_prod_cat_wid = S.unspsc_inv_prod_cat_wid,
    T.commodity_name = S.commodity_name,
    T.commodity_uom_name = S.commodity_uom_name,
    T.ext_store_loc_name = S.ext_store_loc_name,
    T.int_store_loc_name = S.int_store_loc_name,
    T.issue_uom_name = S.issue_uom_name,
    T.loading_type_name = S.loading_type_name,
    T.lot_size_name = S.lot_size_name,
    T.mfg_uom_name = S.mfg_uom_name,
    T.mrp_grp_name = S.mrp_grp_name,
    T.mrp_profile_name = S.mrp_profile_name,
    T.mrp_type_name = S.mrp_type_name,
    T.planner_name = S.planner_name,
    T.primary_uom_name = S.primary_uom_name,
    T.procurement_type_name = S.procurement_type_name,
    T.profit_center_name = S.profit_center_name,
    T.spc_proc_type_name = S.spc_proc_type_name,
    T.status_code = S.status_code,
    T.w_status_code = S.w_status_code,
    T.product_type_code = S.product_type_code,
    T.make_buy_ind = S.make_buy_ind,
    T.fixed_lead_time = S.fixed_lead_time,
    T.variable_lead_time = S.variable_lead_time,
    T.cumulative_total_lead_time = S.cumulative_total_lead_time,
    T.postprocessing_lead_time = S.postprocessing_lead_time,
    T.preprocessing_lead_time = S.preprocessing_lead_time,
    T.process_quality_enabled_flg = S.process_quality_enabled_flg,
    T.x_price_sequence = S.x_price_sequence,
    T.x_organization_name = S.x_organization_name,
    T.x_product_desc = S.x_product_desc,
    T.x_uom_desc = S.x_uom_desc,
    T.x_inv_item_flg = S.x_inv_item_flg,
    T.x_stock_item_flg = S.x_stock_item_flg,
    T.x_trans_flg = S.x_trans_flg,
    T.x_rev_flg = S.x_rev_flg,
    T.x_cost_flg = S.x_cost_flg,
    T.x_gcoa_acct = S.x_gcoa_acct,
    T.x_gcoa_prod = S.x_gcoa_prod,
    T.x_tax_cat = S.x_tax_cat,
    T.organization_id = S.organization_id,
    T.x_gcoa_loc_acct = S.x_gcoa_loc_acct,
    T.effective_to_dt = to_timestamp('3714-01-01 00:00:00', 'yyyy-MM-dd HH:mm:ss'),
    T.current_flg = 'Y',
    T.w_update_dt = current_timestamp(),
    T.etl_proc_wid = (SELECT BIAPPS_ETL_PROC_WID FROM v_bi_apps_params)
WHEN NOT MATCHED AND S.ind_update = 'I' THEN
  INSERT (
    product_wid,
    inventory_org_wid,
    plant_loc_wid,
    product_num,
    abc_ind,
    planner_code,
    procurement_type_code,
    spc_proc_type_code,
    buyer_code,
    buyer_name,
    commodity_code,
    commodity_uom_code,
    profit_center_num,
    reorder_point,
    safety_stock_level,
    min_lot_size,
    max_lot_size,
    fixed_lot_size,
    max_stock_level,
    lot_ordering_cost,
    mrp_time_fence,
    ext_procure_time,
    internal_mfg_time,
    max_storage_days,
    mrp_profile_code,
    mrp_type_code,
    mrp_grp_code,
    lot_size_code,
    backflush_ind,
    qa_inspect_ind,
    repetitive_mfg_ind,
    bulk_item_ind,
    forecast_period,
    mfg_uom_code,
    issue_uom_code,
    manufacturing_place,
    loading_type_code,
    int_store_loc_code,
    ext_store_loc_code,
    active_flg,
    created_by_wid,
    changed_by_wid,
    created_on_dt,
    changed_on_dt,
    aux1_changed_on_dt,
    aux2_changed_on_dt,
    aux3_changed_on_dt,
    aux4_changed_on_dt,
    src_eff_from_dt,
    src_eff_to_dt,
    effective_from_dt,
    delete_flg,
    datasource_num_id,
    integration_id,
    tenant_id,
    x_custom,
    inv_prod_cat1,
    inv_prod_cat2,
    inv_prod_cat3,
    inv_prod_cat4,
    inv_prod_cat5,
    inv_prod_cat6,
    inv_prod_cat7,
    inv_prod_cat8,
    inv_prod_cat9,
    inv_prod_cat10,
    inv_prod_cat1_wid,
    inv_prod_cat2_wid,
    inv_prod_cat3_wid,
    inv_prod_cat4_wid,
    inv_prod_cat5_wid,
    inv_prod_cat6_wid,
    inv_prod_cat7_wid,
    inv_prod_cat8_wid,
    inv_prod_cat9_wid,
    inv_prod_cat10_wid,
    invoiceable_item_flag,
    invoice_enabled_flag,
    primary_uom_code,
    c_primary_uom_code,
    unspsc_code,
    unspsc_inv_prod_cat_wid,
    commodity_name,
    commodity_uom_name,
    ext_store_loc_name,
    int_store_loc_name,
    issue_uom_name,
    loading_type_name,
    lot_size_name,
    mfg_uom_name,
    mrp_grp_name,
    mrp_profile_name,
    mrp_type_name,
    planner_name,
    primary_uom_name,
    procurement_type_name,
    profit_center_name,
    spc_proc_type_name,
    status_code,
    w_status_code,
    product_type_code,
    make_buy_ind,
    fixed_lead_time,
    variable_lead_time,
    cumulative_total_lead_time,
    postprocessing_lead_time,
    preprocessing_lead_time,
    process_quality_enabled_flg,
    x_price_sequence,
    x_organization_name,
    x_product_desc,
    x_uom_desc,
    x_inv_item_flg,
    x_stock_item_flg,
    x_trans_flg,
    x_rev_flg,
    x_cost_flg,
    x_gcoa_acct,
    x_gcoa_prod,
    x_tax_cat,
    organization_id,
    x_gcoa_loc_acct,
    w_insert_dt,
    w_update_dt,
    etl_proc_wid,
    current_flg,
    effective_to_dt
  )
  VALUES (
    S.product_wid,
    S.inventory_org_wid,
    S.plant_loc_wid,
    S.product_num,
    S.abc_ind,
    S.planner_code,
    S.procurement_type_code,
    S.spc_proc_type_code,
    S.buyer_code,
    S.buyer_name,
    S.commodity_code,
    S.commodity_uom_code,
    S.profit_center_num,
    S.reorder_point,
    S.safety_stock_level,
    S.min_lot_size,
    S.max_lot_size,
    S.fixed_lot_size,
    S.max_stock_level,
    S.lot_ordering_cost,
    S.mrp_time_fence,
    S.ext_procure_time,
    S.internal_mfg_time,
    S.max_storage_days,
    S.mrp_profile_code,
    S.mrp_type_code,
    S.mrp_grp_code,
    S.lot_size_code,
    S.backflush_ind,
    S.qa_inspect_ind,
    S.repetitive_mfg_ind,
    S.bulk_item_ind,
    S.forecast_period,
    S.mfg_uom_code,
    S.issue_uom_code,
    S.manufacturing_place,
    S.loading_type_code,
    S.int_store_loc_code,
    S.ext_store_loc_code,
    S.active_flg,
    S.created_by_wid,
    S.changed_by_wid,
    S.created_on_dt,
    S.changed_on_dt,
    S.aux1_changed_on_dt,
    S.aux2_changed_on_dt,
    S.aux3_changed_on_dt,
    S.aux4_changed_on_dt,
    S.src_eff_from_dt,
    S.src_eff_to_dt,
    S.effective_from_dt,
    S.delete_flg,
    S.datasource_num_id,
    S.integration_id,
    S.tenant_id,
    S.x_custom,
    S.inv_prod_cat1,
    S.inv_prod_cat2,
    S.inv_prod_cat3,
    S.inv_prod_cat4,
    S.inv_prod_cat5,
    S.inv_prod_cat6,
    S.inv_prod_cat7,
    S.inv_prod_cat8,
    S.inv_prod_cat9,
    S.inv_prod_cat10,
    S.inv_prod_cat1_wid,
    S.inv_prod_cat2_wid,
    S.inv_prod_cat3_wid,
    S.inv_prod_cat4_wid,
    S.inv_prod_cat5_wid,
    S.inv_prod_cat6_wid,
    S.inv_prod_cat7_wid,
    S.inv_prod_cat8_wid,
    S.inv_prod_cat9_wid,
    S.inv_prod_cat10_wid,
    S.invoiceable_item_flag,
    S.invoice_enabled_flag,
    S.primary_uom_code,
    S.c_primary_uom_code,
    S.unspsc_code,
    S.unspsc_inv_prod_cat_wid,
    S.commodity_name,
    S.commodity_uom_name,
    S.ext_store_loc_name,
    S.int_store_loc_name,
    S.issue_uom_name,
    S.loading_type_name,
    S.lot_size_name,
    S.mfg_uom_name,
    S.mrp_grp_name,
    S.mrp_profile_name,
    S.mrp_type_name,
    S.planner_name,
    S.primary_uom_name,
    S.procurement_type_name,
    S.profit_center_name,
    S.spc_proc_type_name,
    S.status_code,
    S.w_status_code,
    S.product_type_code,
    S.make_buy_ind,
    S.fixed_lead_time,
    S.variable_lead_time,
    S.cumulative_total_lead_time,
    S.postprocessing_lead_time,
    S.preprocessing_lead_time,
    S.process_quality_enabled_flg,
    S.x_price_sequence,
    S.x_organization_name,
    S.x_product_desc,
    S.x_uom_desc,
    S.x_inv_item_flg,
    S.x_stock_item_flg,
    S.x_trans_flg,
    S.x_rev_flg,
    S.x_cost_flg,
    S.x_gcoa_acct,
    S.x_gcoa_prod,
    S.x_tax_cat,
    S.organization_id,
    S.x_gcoa_loc_acct,
    current_timestamp(),
    current_timestamp(),
    (SELECT BIAPPS_ETL_PROC_WID FROM v_bi_apps_params),
    S.current_flg,
    S.effective_to_dt
  );

## Update ETL Load Dates (SCEN_TASK_NO {360})

In [ ]:
%sql
UPDATE
  workspace.prxbi_dw.w_etl_load_dates
SET
  target_table_name = 'W_INVENTORY_PRODUCT_D',
  etl_proc_wid = (SELECT BIAPPS_ETL_PROC_WID FROM v_bi_apps_params),
  load_plan_id = (SELECT BIAPPS_EXECUTION_ID FROM v_bi_apps_params),
  wip_load_start_date = CAST(current_timestamp() - INTERVAL (SELECT BIAPPS_PRUNE_DAYS FROM v_bi_apps_params) DAY AS DATE),
  etl_load_date = current_timestamp(),
  committed = (CASE WHEN (SELECT BIAPPS_IS_INCREMENTAL FROM v_bi_apps_params) = 'Y' THEN '1' ELSE '0' END)
WHERE
  datasource_num_id = (SELECT BIAPPS_DATASOURCE_NUM_ID FROM v_bi_apps_params)
  AND package_name = 'SILOS_SIL_INVENTORYPRODUCTDIMENSION'
  AND etl_usage_code = (SELECT BIAPPS_ETL_USAGE_CODE FROM v_bi_apps_params);

## Insert ETL Load Dates Log (SCEN_TASK_NO {370})

In [ ]:
%sql
INSERT INTO workspace.prxbi_dw.w_etl_load_dates_log (
  datasource_num_id,
  package_name,
  target_table_name,
  etl_usage_code,
  etl_proc_wid,
  load_plan_id,
  session_id,
  wip_load_start_date,
  last_max_date,
  etl_load_date,
  committed
)
SELECT
  datasource_num_id,
  package_name,
  target_table_name,
  etl_usage_code,
  etl_proc_wid,
  load_plan_id,
  CAST((SELECT ODI_SESS_NO FROM v_bi_apps_params) AS BIGINT) AS session_id,
  wip_load_start_date,
  last_max_date,
  etl_load_date,
  committed
FROM
  workspace.prxbi_dw.w_etl_load_dates
WHERE
  datasource_num_id = (SELECT BIAPPS_DATASOURCE_NUM_ID FROM v_bi_apps_params)
  AND package_name = 'SILOS_SIL_INVENTORYPRODUCTDIMENSION'
  AND etl_usage_code = (SELECT BIAPPS_ETL_USAGE_CODE FROM v_bi_apps_params);

## Cleanup

In [ ]:
%sql
-- SCEN_TASK_NO {420}: Drop flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_inventory_product_d_flow;

In [ ]:
%sql
-- SCEN_TASK_NO {440}: Conditionally drop error table if empty
CREATE OR REPLACE TEMPORARY VIEW v_error_count AS
SELECT COUNT(*) AS count_rows FROM workspace.prxbi_dw.e_inventory_product_d;

DROP TABLE IF EXISTS workspace.prxbi_dw.e_inventory_product_d WHERE (SELECT count_rows FROM v_error_count) = 0;

## Conversion Notes and Manual Actions Required

1.  **ROW_WID Handling**: The `ROW_WID` column in `workspace.prxbi_dw.w_inventory_product_d` is assumed to be a `BIGINT GENERATED ALWAYS AS IDENTITY` column. Thus, it has been removed from the `INSERT` column list and values in the final MERGE statement (SCEN_TASK_NO {330} & {350} combined). If `ROW_WID` is intended to be explicitly managed, the DDL for `W_INVENTORY_PRODUCT_D` needs to be `BIGINT GENERATED BY DEFAULT AS IDENTITY`, and the MERGE statement adjusted to include `S.row_wid` or a new `uuid_short()` as value.
2.  **Target Table DDL**: The DDL for the target table `workspace.prxbi_dw.w_inventory_product_d` was not present in the source ODI file. It is assumed to already exist with appropriate column types (e.g., `BIGINT` for WIDs, `DOUBLE` for numeric fields with unspecified precision, `TIMESTAMP` for dates/timestamps, `STRING` for all VARCHAR2/CHAR/CLOB equivalent types) and `ROW_WID` as a `GENERATED ALWAYS AS IDENTITY` column. Manual creation or verification of this DDL is required.
3.  **W_ETL_LOAD_DATES_LOG DDL**: The DDL for `workspace.prxbi_dw.w_etl_load_dates_log` was not provided in the source file. It is assumed to exist with compatible data types. Manual creation or verification of this DDL is required.
4.  **Lookup Table DDLs**: DDLs for `w_ora_invitem_category_tmp`, `w_prod_cat_dh`, `w_inventory_product_ds`, `w_busn_location_d`, `w_int_org_d`, `w_product_d`, `w_user_d`, `w_domain_member_map_g`, `w_domain_member_g` were not provided. These tables must exist in `workspace.prxbi_dw` with compatible Spark SQL data types and column names.
5.  **Parameter `ODI_SESS_NO`**: This parameter was not explicitly defined in the source SQL but is typically used in ODI environments. It has been added as a widget and used for the `session_id` in `W_ETL_LOAD_DATES_LOG` for completeness.
6.  **`TO_DATE` '3714-01-01'**: The Oracle `TO_DATE('01/01/3714 00:00:00', 'MM/DD/YYYY HH24:MI:SS')` for `EFFECTIVE_TO_DT` has been converted to `to_timestamp('3714-01-01 00:00:00', 'yyyy-MM-dd HH:mm:ss')`. Databricks TIMESTAMP type typically supports dates well into the future, but it's good to be aware of this extreme date value.
